<a href="https://colab.research.google.com/github/simon-clematide/colab-notebooks-for-teaching/blob/main/l05_word_embeddings_companion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L05 Companion: Word and Text Embeddings

This notebook follows the lecture *Word and Text Embeddings*. Each part matches a section of the slides:

1. Counting contexts: co-occurrence counts, PPMI, SVD, cosine similarity
2. Predicting contexts: skip-gram with negative sampling in PyTorch
3. Understanding embedding spaces: neighbours, analogies, bias (pretrained GloVe)
4. From word to text embeddings: mean pooling, weighted averaging, what averaging loses
5. Embeddings in neural models: `nn.Embedding`, frozen vectors, fastText n-grams
6. Outlook: subword tokens and a masked language model

The notebook supports understanding of the lecture; it is not a graded exercise. Each part reproduces a claim from the slides and then probes its assumption.

- **Core** (about 60 minutes): parts 1, 2, the nearest neighbours of part 3, and mean pooling in part 4.
- **Explore** (optional, about 45 minutes): analogies and bias, SIF weighting, part 5, part 6.

Cells marked **Your Turn** are short self-checks. Runs on a Colab CPU; part 3 downloads about 66 MB, part 6 about 450 MB.

In [34]:
# On Colab, gensim is not preinstalled.
!pip install -q gensim
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
np.set_printoptions(precision=2, suppress=True)
torch.manual_seed(0)

## 1. Counting Contexts *(Core)*

The toy corpus of the slides, with a window of $\pm1$. $C_{wc}$ counts how often context $c$ occurs next to word $w$.

In [35]:
corpus = ["I like deep learning .", "I like NLP .", "I enjoy flying ."]
sents = [s.split() for s in corpus]
vocab = ["I", "like", "enjoy", "deep", "learning", "NLP", "flying", "."]
idx = {w: i for i, w in enumerate(vocab)}

def cooccurrence(sents, window=1):
    C = np.zeros((len(vocab), len(vocab)))
    for s in sents:
        for i, w in enumerate(s):
            for j in range(max(0, i - window), min(len(s), i + window + 1)):
                if j != i:
                    C[idx[w], idx[s[j]]] += 1
    return C

C = cooccurrence(sents)
print(C.astype(int))
print("total count N =", C.sum())

[[0 2 1 0 0 0 0 0]
 [2 0 0 1 0 1 0 0]
 [1 0 0 0 0 0 1 0]
 [0 1 0 0 1 0 0 0]
 [0 0 0 1 0 0 0 1]
 [0 1 0 0 0 0 0 1]
 [0 0 1 0 0 0 0 1]
 [0 0 0 0 1 1 1 0]]
total count N = 20.0


### PPMI
$\mathrm{PMI}(w,c)=\log_2\frac{P(w,c)}{P(w)P(c)}$, PPMI clips negative values (and $-\infty$ for unseen pairs) at 0.

In [36]:
def ppmi(C):
    N = C.sum()
    pw = C.sum(axis=1, keepdims=True) / N
    pc = C.sum(axis=0, keepdims=True) / N
    with np.errstate(divide="ignore"):
        pmi = np.log2((C / N) / (pw * pc))
    return np.maximum(pmi, 0)

P_ = ppmi(C)
print("PPMI(I, like)   =", round(P_[idx["I"], idx["like"]], 2))    # 1.74 on the slide
print("PPMI(like, NLP) =", round(P_[idx["like"], idx["NLP"]], 2))  # 1.32

PPMI(I, like)   = 1.74
PPMI(like, NLP) = 1.32


### Cosine similarity

In [37]:
def cos(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

print("cos(like, enjoy)  =", round(cos(C[idx["like"]], C[idx["enjoy"]]), 2))   # 0.58
print("cos(like, flying) =", round(cos(C[idx["like"]], C[idx["flying"]]), 2))  # 0

cos(like, enjoy)  = 0.58
cos(like, flying) = 0.0


**Your Turn.** Compute $\cos(\textit{NLP},\textit{flying})$ on raw counts and on PPMI. Then set `window=2`: which similarities change, and why?

In [38]:
# your code here

### Truncated SVD: dense vectors from the count matrix

In [39]:
U, S, Vt = np.linalg.svd(P_)
k = 2
W = U[:, :k] * S[:k]          # one k-dimensional vector per word
for w in vocab:
    print(f"{w:>9}", W[idx[w]])

        I [-1.16  1.1 ]
     like [-1.31 -0.74]
    enjoy [-1.4  -1.65]
     deep [-1.3  -0.44]
 learning [-1.54  1.14]
      NLP [-1.22  1.13]
   flying [-1.59  1.88]
        . [-1.86 -1.9 ]


## 2. Predicting Contexts: Skip-Gram with Negative Sampling *(Core)*

Each word has a center vector $v_w$ (matrix `V`) and a context vector $u_c$ (matrix `U`). For each observed pair $(w,c)$ we draw $k$ negatives $n_i$ from the unigram distribution raised to $0.75$ and minimize

$$\ell=-\log\sigma(u_c^\top v_w)-\sum_{i=1}^{k}\log\sigma(-u_{n_i}^\top v_w).$$

The toy corpus is far too small for meaningful vectors; the point is the mechanics.

In [40]:
pairs = [(idx[w], idx[s[j]]) for s in sents for i, w in enumerate(s)
         for j in range(max(0, i - 1), min(len(s), i + 2)) if j != i]
counts = torch.tensor(C.sum(axis=1))
noise = counts ** 0.75 / (counts ** 0.75).sum()

d, k = 5, 3
V = nn.Embedding(len(vocab), d)   # center vectors v_w
U = nn.Embedding(len(vocab), d)   # context vectors u_c
opt = torch.optim.SGD(list(V.parameters()) + list(U.parameters()), lr=0.1)

w_ids = torch.tensor([p[0] for p in pairs])
c_ids = torch.tensor([p[1] for p in pairs])
for epoch in range(301):
    neg = torch.multinomial(noise, len(pairs) * k, replacement=True).view(-1, k)
    v = V(w_ids)                                       # (B, d)
    pos = (U(c_ids) * v).sum(-1)                       # u_c^T v_w
    negs = torch.bmm(U(neg), v.unsqueeze(-1)).squeeze(-1)  # u_n^T v_w, (B, k)
    loss = -(F.logsigmoid(pos) + F.logsigmoid(-negs).sum(-1)).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 100 == 0:
        print(epoch, round(loss.item(), 3))

0 5.458
100 2.252
200 2.099
300 1.736


**Your Turn.** Set `k = 0` (no negatives) and train again. What happens to the loss and to the norms of the vectors? Why does SGNS need negatives?

*What to look for.* Without negatives the objective only raises the scores $u_c^\top v_w$ of observed pairs. Nothing pushes unobserved pairs down, so the easiest solution is to make all vectors long and similar. The exact numbers depend on initialization, learning rate, and regularization; the direction does not.

In [41]:
# your code here

## 3. Understanding Embedding Spaces *(Core: neighbours; Explore: analogies, bias)*

Pretrained 50-dimensional GloVe vectors (Wikipedia + Gigaword, 400,000 lowercased words).

In [42]:
import gensim.downloader as api
glove = api.load("glove-wiki-gigaword-50")
glove.most_similar("frog", topn=5)

[('snake', 0.8445407748222351),
 ('ape', 0.7937596440315247),
 ('toad', 0.7907540202140808),
 ('monkey', 0.788359522819519),
 ('spider', 0.7718498110771179)]

### Analogies: why the input words are excluded
`most_similar` excludes the input words. Computing 3CosAdd by hand shows what happens without that rule.

In [43]:
print(glove.most_similar(positive=["king", "woman"], negative=["man"], topn=3))

target = glove["king"] - glove["man"] + glove["woman"]
print(glove.similar_by_vector(target, topn=3))   # no exclusion

[('queen', 0.8523604273796082), ('throne', 0.7664334177970886), ('prince', 0.7592144012451172)]
[('king', 0.8859834671020508), ('queen', 0.8609582185745239), ('daughter', 0.7684512734413147)]


### Senses and bias
Lowercased GloVe merges *Bank* and *bank*: subtracting *banking* surfaces a third sense, the West Bank. Senses are mixed by corpus frequency.

In [44]:
print(glove.most_similar(positive=["bank"], negative=["banking"], topn=5))
print(glove.most_similar(positive=["programmer", "woman"], negative=["man"], topn=5))

[('hebron', 0.5932040214538574), ('mustapha', 0.5873048305511475), ('beit', 0.5835797786712646), ('gajevi', 0.5812846422195435), ('nablus', 0.576242208480835)]
[('prodigy', 0.6731117367744446), ('therapist', 0.6414065361022949), ('psychotherapist', 0.6201493144035339), ('technician', 0.6191534996032715), ('programmers', 0.59874427318573)]


**Your Turn.** Find one analogy that works and one that fails. Try relations such as country–capital, verb tense, and antonyms (`hot`/`cold`). Then compute `glove.similarity("hot", "cold")`: why is it high?

In [45]:
# your code here

## 4. From Word to Text Embeddings *(Core: mean pooling; Explore: SIF)*

Mean pooling: $s=\frac{1}{n}\sum_i e_{w_i}$; compare texts with cosine similarity.

In [46]:
def mean_pool(text, kv=glove):
    toks = [t for t in text.lower().replace(".", " .").split() if t in kv]
    return np.mean([kv[t] for t in toks], axis=0)

A = "The committee approved the proposal."
B = "The board accepted the plan."
Cs = "The committee rejected the proposal."
sA, sB, sC = map(mean_pool, [A, B, Cs])
print("cos(A, B) =", round(cos(sA, sB), 3))
print("cos(A, C) =", round(cos(sA, sC), 3))

d1, d2 = mean_pool("The dog bites the man."), mean_pool("The man bites the dog.")
print("identical:", np.allclose(d1, d2))

cos(A, B) = 0.977
cos(A, C) = 0.991
identical: True


Two different kinds of result. That C is closer to A than B is **empirical**: it holds for these GloVe vectors and may change with other embeddings. That the two *dog/man* sentences get the same vector is **guaranteed**: a sum does not depend on order.

### Explore: weighted averaging and common-component removal (Arora et al. 2017)

$s=\frac{1}{n}\sum_i \frac{a}{a+P(w_i)}e_{w_i}$, then remove the first principal component $u$ over all sentence vectors: $s\leftarrow s-uu^\top s$.

GloVe's vocabulary is sorted by frequency; we approximate $P(w)$ with Zipf's law, $P(w)\propto 1/\mathrm{rank}(w)$.

In [47]:
ranks = np.arange(1, len(glove.index_to_key) + 1)
p_zipf = (1 / ranks) / (1 / ranks).sum()

def sif(texts, a=1e-3, kv=glove):
    S = []
    for text in texts:
        toks = [t for t in text.lower().replace(".", " .").split() if t in kv]
        wts = [a / (a + p_zipf[kv.key_to_index[t]]) for t in toks]
        S.append(np.mean([w * kv[t] for w, t in zip(wts, toks)], axis=0))
    S = np.array(S)
    u = np.linalg.svd(S, full_matrices=False)[2][0]   # first principal direction
    return S - np.outer(S @ u, u)

texts = [A, B, Cs, "The weather is nice today.", "Stocks fell sharply on Monday.",
         "The council passed the budget."]
S = sif(texts)
for t, s in zip(texts[1:], S[1:]):
    print(f"{cos(S[0], s):6.3f}  {t}")

 0.559  The board accepted the plan.
 0.841  The committee rejected the proposal.
-0.587  The weather is nice today.
-0.809  Stocks fell sharply on Monday.
 0.433  The council passed the budget.


**Your Turn.** Compare the rankings of mean pooling and SIF for the same six sentences. Does either one put B above C? What would a representation need in order to separate *approved* from *rejected*?

In [48]:
# your code here

## 5. Embeddings in Neural Models *(Explore)*

An embedding lookup equals a bias-free linear layer applied to a one-hot vector.

In [49]:
emb = nn.Embedding(5, 3)
x = torch.zeros(5); x[1] = 1
print(torch.allclose(emb.weight.T @ x, emb(torch.tensor(1))))   # e_w = E^T x

words = ["the", "committee", "approved", "proposal", "."]
pre = torch.tensor(np.stack([glove[w] for w in words]))
frozen = nn.Embedding.from_pretrained(pre, freeze=True)
print(frozen.weight.requires_grad)

bag = nn.EmbeddingBag.from_pretrained(pre, mode="mean")
ids = torch.tensor([[0, 1, 2, 0, 3, 4]])   # "the committee approved the proposal ."
print(torch.allclose(bag(ids)[0], torch.tensor(mean_pool(A)), atol=1e-6))

True
False
True


## 5. Outlook: Embeddings Inside a Language Model *(Explore)*

So far, we have studied **word2vec** and **GloVe**: word vectors learned
from prediction tasks and then used independently of the training model.

A deep language model such as **BERT** also learns an embedding table.
But the lookup is only the beginning: subsequent neural layers transform
the embeddings using information from the surrounding text.

We investigate two questions:

1. **Where does context enter?** Does *bank* have the same representation
   in *river bank* and *central bank*?
2. **Do word analogies transfer to BERT?** Does the familiar
   `king - man + woman ≈ queen` relationship still hold?

We distinguish three kinds of vectors:

- **Input embedding:** a learned lookup-table row, fixed for each token ID.
- **Layer 0:** the input embedding combined with positional information
  and other embedding-layer operations.
- **Hidden states:** representations computed by BERT's neural layers.

We are inspecting **token representations**, not sentence embeddings.

In [50]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "bert-base-cased"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).eval()

print("Vocabulary size:", len(tok))
print("Hidden dimension:", model.config.hidden_size)
print("Transformer layers:", model.config.num_hidden_layers)

for word in ["bank", "king", "man", "woman", "queen"]:
    print(f"{word:>6} -> {tok.tokenize(word)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vocabulary size: 28996
Hidden dimension: 768
Transformer layers: 12
  bank -> ['bank']
  king -> ['king']
   man -> ['man']
 woman -> ['woman']
 queen -> ['queen']


### Experiment 1: Where Does Context Enter?

Consider three occurrences of *bank*:

1. She sat on the bank of the river.
2. He deposited money at the bank.
3. The bank raised its interest rates.

The first sentence uses the geographical sense; the other two use
the financial sense.

**Predict before running the code:**

- Will the input embedding of *bank* differ between sentences?
- Will its representation after BERT's final layer differ?
- Will the two financial uses necessarily be more similar to each other?

We compare cosine similarities at different depths.

In [59]:
sentences = {
    "river": "She sat on the bank of the river.",
    "money": "He deposited money at the bank.",
    "rates": "The bank raised its interest rates.",
}

bank_id = tok.convert_tokens_to_ids("bank")


def bank_representations(sentence):
    enc = tok(sentence, return_tensors="pt")

    positions = (enc["input_ids"][0] == bank_id).nonzero().flatten()
    assert len(positions) == 1, "Expected exactly one 'bank' token."

    pos = positions.item()

    with torch.no_grad():
        states = model(
            **enc,
            output_hidden_states=True
        ).hidden_states

        # Raw embedding-table row: before positional information
        lookup = model.get_input_embeddings().weight[bank_id].clone()

    return {
        "lookup": lookup,
        0: states[0][0, pos],
        1: states[1][0, pos],
        6: states[6][0, pos],
        12: states[12][0, pos],
    }


vectors = {
    name: bank_representations(sentence)
    for name, sentence in sentences.items()
}


def cosine(a, b):
    return F.cosine_similarity(a, b, dim=0).item()


print("Cosine similarity between occurrences of 'bank'\n")

print(f"{'Stage':<12} {'river–money':>14} {'money–rates':>14}")
print("-" * 42)

for layer in ["lookup", 0, 1, 6, 12]:
    rm = cosine(vectors["river"][layer], vectors["money"][layer])
    mr = cosine(vectors["money"][layer], vectors["rates"][layer])

    print(f"{str(layer):<12} {rm:>14.3f} {mr:>14.3f}")

Cosine similarity between occurrences of 'bank'

Stage           river–money    money–rates
------------------------------------------
lookup                1.000          1.000
0                     0.975          0.964
1                     0.885          0.924
6                     0.555          0.759
12                    0.693          0.844


**What do we observe?**

The lookup-table vector is identical in every sentence. It depends
only on the token ID.

After the neural transformations, the representations can differ
because BERT processes the surrounding text.

However, cosine similarity is **not a direct measure of word-sense
disambiguation**. The financial examples need not be closest at every
layer.

**Your Turn.**

Add another sentence containing *bank*. Compare its representation
with the three examples.

At which layers does its representation become more similar to the
occurrence you consider semantically closest?

Why can a static GloVe lookup not make this distinction?

### Experiment 2: Do Word Analogies Transfer to BERT?

Earlier, we investigated word analogies with GloVe:

\[
v_{\text{king}} - v_{\text{man}} + v_{\text{woman}}
\approx v_{\text{queen}}
\]

This relationship can be tested using **3CosAdd**.

But BERT's hidden representations are trained for a different purpose.
They are produced by a neural network, and their geometry can change
between layers.

**Prediction:** Should the analogy work better with BERT because BERT
is a more powerful language model?

We compare three representations:

1. The raw input embedding table.
2. An intermediate BERT layer.
3. The final BERT layer.

To keep the comparison simple, we present each word in isolation,
surrounded by BERT's special tokens.

This is a small diagnostic experiment, not a full analogy benchmark.

In [60]:
def isolated_word_vector(word, layer):
    """
    Return a vector for a single-token word.

    layer='lookup': raw embedding-table row
    layer=0:        embedding-layer output
    layer=1..12:    Transformer hidden states
    """
    enc = tok(word, return_tensors="pt")

    token_ids = tok.encode(word, add_special_tokens=False)

    assert len(token_ids) == 1, (
        f"{word!r} is not a single BERT token: {tok.tokenize(word)}"
    )

    token_id = token_ids[0]

    if layer == "lookup":
        return model.get_input_embeddings().weight[token_id].detach()

    positions = (enc["input_ids"][0] == token_id).nonzero().flatten()
    assert len(positions) == 1

    pos = positions.item()

    with torch.no_grad():
        states = model(
            **enc,
            output_hidden_states=True
        ).hidden_states

    return states[layer][0, pos]


words = ["man", "king", "woman", "queen"]

for layer in ["lookup", 1, 6, 12]:
    vecs = {
        word: isolated_word_vector(word, layer)
        for word in words
    }

    print(
        f"{str(layer):>6}: "
        f"{len(vecs)} vectors, "
        f"dimension {vecs['king'].numel()}"
    )

lookup: 4 vectors, dimension 768
     1: 4 vectors, dimension 768
     6: 4 vectors, dimension 768
    12: 4 vectors, dimension 768


In [62]:
# Candidate words for the analogy: king - man + woman ≈ queen
candidate_words = [
    "queen", "prince", "princess", "duke", "duchess",
    "father", "mother", "boy", "girl", "actor", "actress",
]

# Restrict the experiment to single-token words.
candidates = [
    w for w in candidate_words
    if len(tok.encode(w, add_special_tokens=False)) == 1
]

print("Candidates:", candidates)


def evaluate_analogy(layer):
    all_words = ["man", "king", "woman"] + candidates

    vecs = {
        word: isolated_word_vector(word, layer)
        for word in set(all_words)
    }

    # king - man + woman
    target = vecs["king"] - vecs["man"] + vecs["woman"]

    # Exclude input words, as in 3CosAdd.
    excluded = {"king", "man", "woman"}

    scores = {
        word: cosine(target, vecs[word])
        for word in candidates
        if word not in excluded
    }

    ranking = sorted(
        scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    print(f"\n--- {layer} ---")

    for rank, (word, score) in enumerate(ranking[:5], start=1):
        marker = " <-- expected" if word == "queen" else ""
        print(f"{rank:>2}. {word:<12} {score:.3f}{marker}")

    queen_rank = next(
        i for i, (word, _) in enumerate(ranking, start=1)
        if word == "queen"
    )

    print(f"Rank of queen: {queen_rank}/{len(ranking)}")


for layer in ["lookup", 1, 6, 12]:
    evaluate_analogy(layer)

Candidates: ['queen', 'prince', 'princess', 'duke', 'father', 'mother', 'boy', 'girl', 'actor', 'actress']

--- lookup ---
 1. queen        0.618 <-- expected
 2. princess     0.464
 3. prince       0.401
 4. duke         0.312
 5. girl         0.309
Rank of queen: 1/10

--- 1 ---
 1. queen        0.733 <-- expected
 2. princess     0.565
 3. prince       0.512
 4. duke         0.480
 5. girl         0.442
Rank of queen: 1/10

--- 6 ---
 1. queen        0.874 <-- expected
 2. prince       0.774
 3. princess     0.749
 4. duke         0.719
 5. mother       0.679
Rank of queen: 1/10

--- 12 ---
 1. prince       0.902
 2. queen        0.890 <-- expected
 3. princess     0.879
 4. actress      0.849
 5. duke         0.839
Rank of queen: 2/10


**What do we learn?**

The analogy may behave differently across BERT's input embeddings
and hidden layers.

A model that is more powerful at language understanding does not
necessarily produce better vectors for **3CosAdd**.

Three cautions:

1. We tested only one analogy and a small candidate set.
2. BERT's hidden states depend on the input and the layer.
3. A failed analogy does not imply that BERT lacks the corresponding
   linguistic knowledge.

**Your Turn.**

Try another analogy, such as:

- `paris - france + germany → berlin`
- `walked - walk + play → played`

Which analogy works best at which layer?

Does the result support a general conclusion about BERT?

### Takeaway

**word2vec:** Learn word vectors and reuse them independently.

**BERT:** Learn input embeddings together with a network that
transforms them into contextual representations.

The interesting representations are no longer necessarily the
lookup-table vectors.

In [64]:
# Additional analogies to test across BERT layers
analogies = [
    {
        "name": "Geography",
        "a": "France",
        "b": "Paris",
        "c": "Germany",
        "expected": "Berlin",
        "candidates": [
            "Berlin", "London", "Rome", "Madrid",
            "Vienna", "Paris", "Munich", "Hamburg",
            "France", "Germany",
        ],
    },
    {
        "name": "Past tense",
        "a": "walk",
        "b": "walked",
        "c": "play",
        "expected": "played",
        "candidates": [
            "played", "playing", "plays", "play",
            "walked", "walking", "walk",
            "worked", "talked", "wanted",
        ],
    },
]


def test_analogy(example, layer):
    a, b, c = example["a"], example["b"], example["c"]

    # Keep only single-token candidates.
    candidates = [
        w for w in example["candidates"]
        if len(tok.encode(w, add_special_tokens=False)) == 1
        and w not in {a, b, c}
    ]

    # Check that the analogy inputs and expected answer are usable.
    required = [a, b, c, example["expected"]]
    if any(len(tok.encode(w, add_special_tokens=False)) != 1
           for w in required):
        print(f"Skipping {example['name']}: subword tokenization")
        return

    words = set([a, b, c] + candidates)
    vecs = {
        w: isolated_word_vector(w, layer)
        for w in words
    }

    # b - a + c
    target = vecs[b] - vecs[a] + vecs[c]

    ranking = sorted(
        [(w, cosine(target, vecs[w])) for w in candidates],
        key=lambda item: item[1],
        reverse=True
    )

    expected = example["expected"]
    rank = next(
        (i for i, (w, _) in enumerate(ranking, 1) if w == expected),
        None
    )

    print(f"\n{example['name']} — layer {layer}")
    print(f"{b} - {a} + {c} ≈ {expected}")

    for i, (word, score) in enumerate(ranking[:5], 1):
        marker = " <-- expected" if word == expected else ""
        print(f"{i:>2}. {word:<12} {score:.3f}{marker}")

    print(f"Rank of {expected}: {rank}/{len(ranking)}")


for example in analogies:
    for layer in ["lookup", 1, 6, 12]:
        test_analogy(example, layer)


Geography — layer lookup
Paris - France + Germany ≈ Berlin
 1. Berlin       0.601 <-- expected
 2. Munich       0.488
 3. Vienna       0.451
 4. Hamburg      0.438
 5. London       0.395
Rank of Berlin: 1/7

Geography — layer 1
Paris - France + Germany ≈ Berlin
 1. Berlin       0.712 <-- expected
 2. Munich       0.634
 3. Vienna       0.555
 4. London       0.553
 5. Hamburg      0.538
Rank of Berlin: 1/7

Geography — layer 6
Paris - France + Germany ≈ Berlin
 1. Berlin       0.817 <-- expected
 2. Munich       0.767
 3. London       0.751
 4. Vienna       0.730
 5. Hamburg      0.728
Rank of Berlin: 1/7

Geography — layer 12
Paris - France + Germany ≈ Berlin
 1. Berlin       0.935 <-- expected
 2. Munich       0.931
 3. Hamburg      0.922
 4. London       0.911
 5. Madrid       0.900
Rank of Berlin: 1/7

Past tense — layer lookup
walked - walk + play ≈ played
 1. played       0.694 <-- expected
 2. plays        0.614
 3. playing      0.522
 4. worked       0.249
 5. wanted       0.1